In [ ]:
from datetime import date
from pandas import json_normalize
import pandas as pd
pd.set_option('display.max_columns', None)

In [ ]:
df_work_raw = catalog.load('raw/openalex/work#parquet')

In [ ]:
df_work_raw = df_work_raw.convert_dtypes()
df_work_raw = df_work_raw.loc[:,['id','locations']]

In [ ]:
df_work_raw.info()

In [ ]:
df_work_raw

In [ ]:
df_work_authorship = df_work_raw.explode('locations').reset_index(drop=True)

In [ ]:
def openalex_load_work_location(df_work_raw):

    df_work_raw.rename(columns={'id':'work_id'}, inplace=True)
    df_work_location = df_work_raw.explode('locations').reset_index(drop=True)

    df_work_location = pd.concat([df_work_location['work_id'], json_normalize(df_work_location['locations'])], axis=1)

    df_work_location.columns = df_work_location.columns.str.replace('.', '_')

    df_work_location = df_work_location[[
        'work_id','id', 
        'source_id', 'source_display_name', 'source_is_core', 'source_type',
        'source_host_organization', 'source_host_organization_name',
        'is_accepted', 'is_oa', 'is_published', 'landing_page_url',
        'license', 'license_id', 'pdf_url', 'version',
        # 'source_host_organization_lineage', 'source_host_organization_lineage_names', 'source_issn',
        'source_is_in_doaj', 'source_is_oa', 'source_issn_l'
    ]]

    df_work_location['_load_datetime'] = date.today()

    return df_work_location


In [ ]:
stage_work_location = openalex_load_work_location(df_work_raw)
stage_work_location